# CatBoost на расширенной разметке

Сравниваем CatBoost и лучшую конфигурацию L2 из `model_tuning.ipynb` на одной таблице признаков. Запускать из корня проекта. Подбор CatBoost занимает несколько минут.

## 1. Загрузить признаки и метки

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

In [ ]:
FEATURES_PATH = Path("data_by_dewallen/features.csv")
VERIFIED_PATH = Path("data/verified.csv")
EXTENDED_VERIFIED_PATH = Path("extend_verified.csv")
LISTINGS_PATH = Path("data/listings.csv")

DEPTHS = [3, 5]
ITERATIONS = [80, 180]
EXTRA_WEIGHT_SCALES = [0.0, 0.1, 0.25, 0.5, 1.0]
THRESHOLDS = np.round(np.arange(0.20, 0.71, 0.01), 2)
CV_SEEDS = [11, 23, 42]
INNER_SEED = 42
L2_C = 0.3
L2_EXTRA_SCALE = 0.1

In [ ]:
features = pd.read_csv(FEATURES_PATH, index_col="listing_id")
verified = pd.read_csv(VERIFIED_PATH, usecols=["listing_id", "is_deal"], index_col="listing_id")
listings = pd.read_csv(LISTINGS_PATH, usecols=["listing_id", "close_type"]).drop_duplicates()
listings = listings.set_index("listing_id")

assert listings.index.is_unique and features.index.is_unique
assert set(features.index) == set(listings.index), "Пересчитайте признаки для всех объявлений"
assert verified.index.is_unique, "Повторяются ID в проверенной разметке"
assert verified.index.isin(features.index).all()
assert verified["is_deal"].isin([0, 1]).all(), "Некорректные метки в verified.csv"
assert features.columns.is_unique

feature_columns = features.columns.difference(["seller_id", "is_deal", "weight"], sort=False)
X = features[feature_columns]
y = verified["is_deal"]

print(f"Объявлений в матрице: {len(X)}")
print(f"Признаков: {X.shape[1]}")
print(f"Объявлений с проверенными метками: {len(y)}")

### Веса разметки

Проверенным ответам даём вес 1.0. Остальные строки берём из `extend_verified.csv` и умножаем их исходный вес на множитель. Множитель 1.0 проверяет исходные веса 0.8 и 1.0. При множителе 0 модель обучается на одних проверенных метках.

In [ ]:
extended = pd.read_csv(
    EXTENDED_VERIFIED_PATH,
    usecols=["listing_id", "is_deal", "weight"],
    index_col="listing_id",
)
assert extended.index.is_unique, "Повторяются ID в расширенной разметке"
assert extended.index.notna().all(), "В расширенной разметке есть пустые ID"

extra_labels = extended.loc[~extended.index.isin(verified.index)]
assert extra_labels.index.isin(X.index).all(), "Для дополнительных меток нет признаков"
assert extra_labels["is_deal"].isin([0, 1]).all(), "Некорректные дополнительные метки"
assert extra_labels["weight"].gt(0).all() and np.isfinite(extra_labels["weight"]).all()

print("Дополнительных меток:", len(extra_labels))
display(extra_labels.groupby(["is_deal", "weight"]).size())

In [ ]:
labels_by_scale = {
    scale: extra_labels.iloc[:0] if scale == 0
    else extra_labels.assign(weight=extra_labels["weight"] * scale)
    for scale in EXTRA_WEIGHT_SCALES
}

blocked_ids = listings.index[listings["close_type"] == "заблокировано"]
categories = features["category"].loc[y.index]
assert categories.notna().all(), "Для проверки долей у объявления должна быть категория"

print("Конфигураций CatBoost:", len(DEPTHS) * len(ITERATIONS) * len(EXTRA_WEIGHT_SCALES))

## 2. Подготовить CatBoost и контрольную L2

CatBoost сам обрабатывает числовые пропуски и категориальные признаки. Чтобы сравнить 80 и 180 деревьев, обучаем модель на 180 и берём прогнозы на обоих шагах. Для L2 используем ту же предобработку, что в предыдущем подборе.

In [ ]:
X_cat = X.copy()
categorical_columns = X_cat.select_dtypes(exclude=["number", "bool"]).columns.tolist()
numeric_columns = X_cat.columns.difference(categorical_columns, sort=False)

X_cat[numeric_columns] = X_cat[numeric_columns].replace([np.inf, -np.inf], np.nan)
for column in categorical_columns:
    X_cat[column] = X_cat[column].astype("string").fillna("unknown")

print("Категориальных признаков:", len(categorical_columns))


def fit_catboost(depth, train_y, labels, iterations):
    train_labels = pd.concat([
        train_y.to_frame().assign(weight=1.0),
        labels,
    ])
    train_pool = Pool(
        X_cat.loc[train_labels.index],
        label=train_labels["is_deal"],
        weight=train_labels["weight"],
        cat_features=categorical_columns,
    )
    model = CatBoostClassifier(
        depth=depth,
        iterations=iterations,
        learning_rate=0.08,
        l2_leaf_reg=5,
        border_count=32,
        loss_function="Logloss",
        thread_count=4,
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
    )
    return model.fit(train_pool)

In [ ]:
def clean_numbers(frame):
    return frame.astype(float).replace([np.inf, -np.inf], np.nan)


def clean_categories(frame):
    strings = frame.astype("string")
    return strings.astype(object).where(strings.notna(), np.nan)


numeric_pipeline = make_pipeline(
    FunctionTransformer(clean_numbers, feature_names_out="one-to-one"),
    SimpleImputer(strategy="median"),
    StandardScaler(),
)
categorical_pipeline = make_pipeline(
    FunctionTransformer(clean_categories, feature_names_out="one-to-one"),
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore"),
)
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, make_column_selector(dtype_include=["number", "bool"])),
    ("categorical", categorical_pipeline, make_column_selector(dtype_exclude=["number", "bool"])),
])
model = make_pipeline(
    preprocessor,
    LogisticRegression(C=L2_C, max_iter=2000),
)

def fit_model(model, X, y_train, extra_labels):
    train_labels = pd.concat([
        y_train.to_frame().assign(weight=1.0),
        extra_labels,
    ])
    X_fit = X.loc[train_labels.index]
    return clone(model).fit(
        X_fit,
        train_labels["is_deal"],
        logisticregression__sample_weight=train_labels["weight"],
    )

## 3. Подобрать пороги и получить отложенные прогнозы

Делим `verified.csv` на те же три разбиения по пять внешних фолдов, что и в предыдущем эксперименте. Внутри каждой обучающей части используем четыре фолда для выбора порога по F1. Дополнительные метки участвуют в обучении. Для заблокированных объявлений ставим нулевую вероятность сделки.

In [ ]:
def threshold_by_f1(train_y, probabilities):
    predicted = probabilities[:, None] > THRESHOLDS
    actual = train_y.to_numpy() == 1
    true_positives = (predicted & actual[:, None]).sum(axis=0)
    f1_scores = 2 * true_positives / (predicted.sum(axis=0) + actual.sum())

    best_indices = np.flatnonzero(f1_scores == f1_scores.max())
    best = best_indices[np.abs(THRESHOLDS[best_indices] - 0.5).argmin()]
    return THRESHOLDS[best], f1_scores[best]


def catboost_probabilities(model, test_pool, test_ids, iterations):
    probabilities = model.predict_proba(test_pool, ntree_end=iterations, thread_count=4)[:, 1]
    probabilities[test_ids.isin(blocked_ids)] = 0
    return probabilities


def inner_catboost_predictions(depth, train_y, labels):
    inner_cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=INNER_SEED)
    probabilities = {iterations: np.zeros(len(train_y)) for iterations in ITERATIONS}

    for train, test in inner_cv.split(train_y, train_y):
        fitted = fit_catboost(depth, train_y.iloc[train], labels, max(ITERATIONS))
        test_ids = train_y.iloc[test].index
        test_pool = Pool(X_cat.loc[test_ids], cat_features=categorical_columns)
        for iterations in ITERATIONS:
            probabilities[iterations][test] = catboost_probabilities(
                fitted, test_pool, test_ids, iterations,
            )

    return probabilities

In [ ]:
def l2_probabilities(fitted, test_ids):
    probabilities = fitted.predict_proba(X.loc[test_ids])[:, 1]
    probabilities[test_ids.isin(blocked_ids)] = 0
    return probabilities


def l2_fold_predictions(train_y, test_ids):
    inner_cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=INNER_SEED)
    probabilities = np.zeros(len(train_y))
    labels = labels_by_scale[L2_EXTRA_SCALE]

    for train, test in inner_cv.split(train_y, train_y):
        fitted = fit_model(model, X, train_y.iloc[train], labels)
        probabilities[test] = l2_probabilities(fitted, train_y.iloc[test].index)

    threshold, _ = threshold_by_f1(train_y, probabilities)
    fitted = fit_model(model, X, train_y, labels)
    return (l2_probabilities(fitted, test_ids) > threshold).astype(int)

In [ ]:
def validation_metrics(predictions):
    shares = pd.DataFrame({"actual": y, "predicted": predictions}, index=y.index)
    shares = shares.join(categories).groupby("category")[["actual", "predicted"]].mean()

    return {
        "F1": f1_score(y, predictions),
        "Precision": precision_score(y, predictions, zero_division=0),
        "Recall": recall_score(y, predictions, zero_division=0),
        "CatShareMAE": 100 * (shares["actual"] - shares["predicted"]).abs().mean(),
    }

In [ ]:
configs = [
    (depth, iterations, scale)
    for depth in DEPTHS
    for iterations in ITERATIONS
    for scale in EXTRA_WEIGHT_SCALES
]
results = []
selection_results = []
selected_configs = []

for seed in CV_SEEDS:
    outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    predictions = {config: np.full(len(y), -1) for config in configs}
    selected_predictions = np.full(len(y), -1)
    baseline_predictions = np.full(len(y), -1)

    for fold, (train, test) in enumerate(outer_cv.split(y, y), start=1):
        train_y, test_ids = y.iloc[train], y.iloc[test].index
        test_pool = Pool(X_cat.loc[test_ids], cat_features=categorical_columns)
        inner_scores = {}

        for depth in DEPTHS:
            for scale, labels in labels_by_scale.items():
                inner_probabilities = inner_catboost_predictions(depth, train_y, labels)
                fitted = fit_catboost(depth, train_y, labels, max(ITERATIONS))

                for iterations, probabilities in inner_probabilities.items():
                    config = (depth, iterations, scale)
                    threshold, inner_f1 = threshold_by_f1(train_y, probabilities)
                    outer_probabilities = catboost_probabilities(fitted, test_pool, test_ids, iterations)
                    predictions[config][test] = (outer_probabilities > threshold).astype(int)
                    inner_scores[config] = (inner_f1, threshold)

        best_config = max(configs, key=lambda config: inner_scores[config][0])
        selected_predictions[test] = predictions[best_config][test]
        baseline_predictions[test] = l2_fold_predictions(train_y, test_ids)
        selected_configs.append({
            "Разбиение": seed, "Фолд": fold,
            "Глубина": best_config[0], "Деревья": best_config[1], "Множитель": best_config[2],
            "Порог": inner_scores[best_config][1],
        })
        print("Проверено разбиение", seed, "| фолд", fold, flush=True)

    assert (selected_predictions >= 0).all() and (baseline_predictions >= 0).all()
    for (depth, iterations, scale), labels in predictions.items():
        assert (labels >= 0).all(), "Для части проверенных объявлений нет прогноза"
        results.append({
            "Разбиение": seed, "Глубина": depth, "Деревья": iterations, "Множитель": scale,
            **validation_metrics(labels),
        })

    selection_results.extend([
        {"Разбиение": seed, "Модель": "CatBoost с подбором", **validation_metrics(selected_predictions)},
        {"Разбиение": seed, "Модель": "L2 C=0.3", **validation_metrics(baseline_predictions)},
    ])

## 4. Сравнить F1 и ошибку долей

Средние считаем по трём разбиениям. В каждом разбиении F1 рассчитан по всем 300 отложенным прогнозам. Лидер рейтинга выбран после просмотра внешних результатов, поэтому отдельно показываем оценку процедуры подбора. В ней глубина, число деревьев, вес и порог выбираются по внутренним фолдам.

In [12]:
results = pd.DataFrame(results)
ranking = (
    results.groupby(["Глубина", "Деревья", "Множитель"], as_index=False)
    .agg(
        F1=("F1", "mean"),
        F1_min=("F1", "min"),
        Precision=("Precision", "mean"),
        Recall=("Recall", "mean"),
        CatShareMAE=("CatShareMAE", "mean"),
    )
    .sort_values(["F1", "CatShareMAE"], ascending=[False, True])
    .reset_index(drop=True)
)

print("Лучшие настройки по F1")
display(ranking.head(10).round(4))

print("Настройка с наименьшей ошибкой долей")
display(ranking.nsmallest(1, "CatShareMAE").round(4))

comparison = pd.DataFrame(selection_results)
print("Внешняя проверка CatBoost с подбором и контрольной L2")
display(comparison.round(4))
display(comparison.drop(columns="Разбиение").groupby("Модель").mean().round(4))
display(pd.DataFrame(selected_configs).round(4))

Лучшие настройки по F1
Настройка с наименьшей ошибкой долей
Внешняя проверка CatBoost с подбором и контрольной L2


,Глубина,Деревья,Множитель,F1,F1_min,Precision,Recall,CatShareMAE
0,5,180,0.10,0.7999,0.7915,0.7793,0.8218,5.5244
1,3,180,0.10,0.7962,0.7917,0.7756,0.8190,6.4704
2,3,180,0.00,0.7934,0.7803,0.7924,0.7960,7.0540
3,3,80,0.25,0.7927,0.7750,0.7735,0.8132,5.9075
4,3,180,0.25,0.7919,0.7860,0.7806,0.8046,6.6799
5,3,80,0.10,0.7890,0.7840,0.7709,0.8103,6.1480
6,5,180,0.25,0.7861,0.7686,0.7771,0.7960,6.0069
7,5,180,0.00,0.7860,0.7603,0.7720,0.8017,5.4098
8,3,180,1.00,0.7854,0.7750,0.7575,0.8161,6.7252
9,3,80,0.00,0.7843,0.7759,0.7904,0.7787,5.2853


,Глубина,Деревья,Множитель,F1,F1_min,Precision,Recall,CatShareMAE
16,5,80,0.0,0.7781,0.7619,0.7803,0.7759,4.9891


,Разбиение,Модель,F1,Precision,Recall,CatShareMAE
0,11,CatBoost с подбором,0.8052,0.8087,0.8017,5.5825
1,11,L2 C=0.3,0.8051,0.7917,0.8190,3.7117
2,23,CatBoost с подбором,0.7881,0.7750,0.8017,5.4953
3,23,L2 C=0.3,0.8250,0.7984,0.8534,3.5984
4,42,CatBoost с подбором,0.7949,0.7881,0.8017,5.7788
5,42,L2 C=0.3,0.8230,0.7874,0.8621,4.6457


,F1,Precision,Recall,CatShareMAE
Модель,,,,
CatBoost с подбором,0.7961,0.7906,0.8017,5.6189
L2 C=0.3,0.8177,0.7925,0.8448,3.9853


,Разбиение,Фолд,Глубина,Деревья,Множитель,Порог
0,11,1,3,80,0.25,0.60
1,11,2,3,180,0.10,0.48
2,11,3,3,180,0.25,0.65
3,11,4,3,80,0.10,0.47
4,11,5,3,80,0.00,0.49
5,23,1,3,180,0.25,0.50
6,23,2,3,80,0.50,0.60
7,23,3,5,80,0.50,0.63
8,23,4,3,180,0.10,0.46
9,23,5,5,180,0.00,0.29


## 5. Обучить лидера по F1

Подбираем один порог по отложенным прогнозам для всей проверенной разметки. Обучаем модель с выбранными параметрами и получаем метки для всех объявлений в `catboost_predictions`.

In [13]:
best = ranking.iloc[0]
depth = int(best["Глубина"])
iterations = int(best["Деревья"])
scale = best["Множитель"]
labels = labels_by_scale[scale]

probabilities = inner_catboost_predictions(depth, y, labels)[iterations]
threshold, _ = threshold_by_f1(y, probabilities)
final_model = fit_catboost(depth, y, labels, iterations)

all_pool = Pool(X_cat, cat_features=categorical_columns)
probabilities = catboost_probabilities(final_model, all_pool, X_cat.index, iterations)
catboost_predictions = X.index.to_frame(index=False).assign(
    is_deal=(probabilities > threshold).astype(int),
)
assert catboost_predictions["listing_id"].is_unique
assert set(catboost_predictions["listing_id"]) == set(listings.index)
assert catboost_predictions.loc[catboost_predictions["listing_id"].isin(blocked_ids), "is_deal"].eq(0).all()

print("Глубина:", depth)
print("Деревьев:", iterations)
print("Множитель весов:", scale)
print("Порог:", round(threshold, 2))
print("Объявлений с прогнозом:", len(catboost_predictions))

Глубина: 5
Деревьев: 180
Множитель весов: 0.1
Порог: 0.39
Объявлений с прогнозом: 15274
